<a href="https://colab.research.google.com/github/bahmedx/733/blob/main/Real_World_AI_Driven_Analytics_Project_Part_2_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Import Libraries**
This section imports the Python libraries required for data manipulation, visualization, preprocessing, model development, optimization, and evaluation. Pandas and NumPy support data processing, Matplotlib and Seaborn support visualization, and scikit-learn supports preprocessing, Random Forest classification, hyperparameter tuning, and model evaluation.

In [ ]:
# CELL 1: Imports and Environment Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn & Imblearn modules
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV

# Handling Imbalance
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Original Model
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')

### **Dataset Exploration and Preprocessing**
This section examines the Diabetes 130-US Hospitals dataset to understand its structure, feature types, missing values, and overall data quality. Missing-value indicators are standardized, duplicate records are assessed, selected high-missing features are removed, and administrative ID mappings are used to improve interpretability.

A binary target variable is created for 30-day readmission, where 1 represents readmission within 30 days and 0 represents later or no readmission. Readable feature labels are also created for exploratory analysis and visualization.

**Upload, Extract, and Load Dataset**

In [ ]:
# CELL 2: Data Upload from Local Machine and Memory Optimization
import os
import zipfile
from google.colab import files

def reduce_mem_usage(df):
    """Iterate through all columns and modify data types to reduce memory usage in Colab."""
    start_mem = df.memory_usage().sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
        else:
            # Convert low-cardinality objects to category
            if df[col].nunique() / len(df) < 0.5:
                df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory usage decreased from {start_mem:.2f} MB to {end_mem:.2f} MB')
    return df

# 1. Prompt user to upload the zip file from local machine
print("Please upload your dataset (.zip) file:")
uploaded = files.upload()

# 2. Get the uploaded filename and extract it
zip_filename = list(uploaded.keys())[0]
extract_dir = 'dataset_extracted'

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
    extracted_files = zip_ref.namelist()

# 3. Find the CSV file in the extracted contents (assumes the main data is a .csv)
csv_filename = [f for f in extracted_files if f.endswith('.csv')][0]
csv_path = os.path.join(extract_dir, csv_filename)

# 4. Load the dataset and apply memory optimization
df = pd.read_csv(csv_path)
df = reduce_mem_usage(df)
print(f"Successfully loaded: {csv_filename}")

In [ ]:
# CELL 3: Basic Preprocessing & Target Definition

# 1. Handle missing values marked as '?'
df.replace('?', np.nan, inplace=True)

# 2. Drop columns with extreme missingness or identifying info (except patient_nbr needed for splitting)
df.drop(columns=['weight', 'payer_code', 'medical_specialty', 'encounter_id'], inplace=True, errors='ignore')

# 3. Create Binary Target: 1 for <30 days readmission, 0 otherwise (if not already created)
if 'readmitted' in df.columns:
    df['target'] = (df['readmitted'] == '<30').astype(int)
    df.drop(columns=['readmitted'], inplace=True)

# 4. Feature Grouping: Mapping ICD-9 diagnosis codes to broader categories
def map_icd9(val):
    if pd.isna(val) or val == '?':
        return 'Missing'

    # Handle diabetes codes directly
    if str(val).startswith('250'):
        return 'Diabetes'

    # Handle supplemental classification codes
    if 'V' in str(val) or 'E' in str(val):
        return 'Other'

    try:
        # Convert to float to evaluate numerical ranges
        num_val = float(val)
        if 390 <= num_val <= 459 or num_val == 785:
            return 'Circulatory'
        elif 460 <= num_val <= 519 or num_val == 786:
            return 'Respiratory'
        elif 520 <= num_val <= 579 or num_val == 787:
            return 'Digestive'
        elif 800 <= num_val <= 999:
            return 'Injury'
        elif 710 <= num_val <= 739:
            return 'Musculoskeletal'
        elif 580 <= num_val <= 629 or num_val == 788:
            return 'Genitourinary'
        elif 140 <= num_val <= 239:
            return 'Neoplasms'
        else:
            return 'Other'
    except ValueError:
        return 'Other'

# Apply the mapping to the diagnosis columns
for col in ['diag_1', 'diag_2', 'diag_3']:
    if col in df.columns:
        df[col] = df[col].apply(map_icd9)
        # Ensure the column is properly typed for the categorical pipeline
        df[col] = df[col].astype('category')

print("Preprocessing complete. Diagnosis codes grouped.")

In [ ]:
# CELL 4: Patient-Level Data Splitting (Fixing Data Leakage)
X = df.drop(columns=['target'])
y = df['target']
groups = df['patient_nbr']

# GroupShuffleSplit ensures the same patient doesn't appear in both train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Drop patient_nbr as it's no longer needed for modeling
X_train = X_train.drop(columns=['patient_nbr'])
X_test = X_test.drop(columns=['patient_nbr'])

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
# CELL 5: Imblearn Pipeline Setup
numeric_features = X_train.select_dtypes(include=['int8', 'int16', 'int32', 'float32', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['category', 'object']).columns.tolist()

# Preprocessing for numerical data
numeric_transformer = SklearnPipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessing for categorical data
categorical_transformer = SklearnPipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Create the imblearn pipeline: Preprocess -> SMOTE -> Classifier
pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])

print("Pipeline setup complete.")

In [ ]:
# CELL 6: Efficient Hyperparameter Tuning (Colab-Optimized Version)
from sklearn.model_selection import RandomizedSearchCV

# Scaled-down parameter distributions to prevent Colab from hanging
param_distributions = {
    'classifier__n_estimators': [50, 100, 150], # Reduced tree count
    'classifier__max_depth': [10, 15],          # Capped depth to prevent memory exhaustion
    'classifier__min_samples_split': [5, 10],
    'classifier__min_samples_leaf': [2, 4],
    'smote__k_neighbors': [3, 5]
}

# Initialize RandomizedSearchCV with fewer iterations
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=5, # Reduced to 5 iterations (15 total fits across 3 CV folds)
    scoring='recall',
    cv=3,
    random_state=42,
    n_jobs=1, # Changed to 1 to prevent multiprocessing deadlocks in Colab
    verbose=3 # Increased verbosity so you can track the exact step it is on
)

# Fit the search model
print("Starting Hyperparameter Tuning. With these settings, it should complete shortly...")
search.fit(X_train, y_train)

# Output the results
print("\n--- Tuning Complete ---")
print(f"Best Hyperparameters: {search.best_params_}")
print(f"Best CV Recall Score: {search.best_score_:.4f}")

# Extract the best trained model for threshold tuning in the next cell
best_model = search.best_estimator_

**Initial Dataset Inspection**

In [ ]:
print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
display(df.dtypes.to_frame("Data Type"))

print("\nDataset Sample:")
display(df.sample(5, random_state=42))

**Data Quality Assessment**

In [ ]:
# Convert UCI missing-value marker to NaN
df = df.replace("?", np.nan)

# Duplicate records
print("Duplicate Rows:", df.duplicated().sum())

# Missing-value summary
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percent": (
        df.isnull().mean() * 100
    ).round(2)
})

missing_summary = (
    missing_summary[
        missing_summary["Missing Count"] > 0
    ]
    .sort_values(
        "Missing Percent",
        ascending=False
    )
)

display(missing_summary)

**Remove High-Missing and Non-Useful Features**

In [ ]:
columns_to_drop = [
    "weight",
    "payer_code",
    "medical_specialty"
]

df = df.drop(columns=columns_to_drop, errors="ignore")

print("Dataset shape after removing selected columns:", df.shape)

**Load UCI ID Mappings**

In [ ]:
# Locate IDS_mapping.csv in the extracted UCI dataset
mapping_path = None

for root, dirs, files_in_dir in os.walk("/content"):
    if "IDS_mapping.csv" in files_in_dir:
        mapping_path = os.path.join(root, "IDS_mapping.csv")
        break

if mapping_path is None:
    raise FileNotFoundError(
        "IDS_mapping.csv was not found in the extracted dataset."
    )

print("Mapping file found:", mapping_path)


# ------------------------------------------------------------
# Read raw mapping file
# ------------------------------------------------------------
mapping_raw = pd.read_csv(
    mapping_path,
    header=None,
    dtype=str
)

# Keep first two columns
mapping_raw = mapping_raw.iloc[:, :2].copy()
mapping_raw.columns = ["id", "description"]

# Clean whitespace
mapping_raw["id"] = mapping_raw["id"].str.strip()
mapping_raw["description"] = (
    mapping_raw["description"]
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# Function to extract an individual mapping section
# ------------------------------------------------------------
mapping_sections = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]


def extract_id_mapping(section_name):

    # Find start of requested section
    section_rows = mapping_raw.index[
        mapping_raw["id"].eq(section_name)
    ]

    if len(section_rows) == 0:
        raise ValueError(
            f"{section_name} was not found in IDS_mapping.csv."
        )

    start = section_rows[0] + 1

    # Find start of next section
    next_sections = mapping_raw.index[
        (mapping_raw.index >= start) &
        (mapping_raw["id"].isin(mapping_sections))
    ]

    end = (
        next_sections[0]
        if len(next_sections) > 0
        else len(mapping_raw)
    )

    section = (
        mapping_raw.loc[
            start:end - 1,
            ["id", "description"]
        ]
        .copy()
    )

    # Convert ID values to numeric
    section["id"] = pd.to_numeric(
        section["id"],
        errors="coerce"
    )

    # Remove headings, blank rows, and non-ID values
    section = (
        section
        .dropna(subset=["id"])
        .assign(
            id=lambda x: x["id"].astype(int)
        )
    )

    # Return reusable dictionary
    return dict(
        zip(
            section["id"],
            section["description"]
        )
    )


# ------------------------------------------------------------
# Create mappings directly from IDS_mapping.csv
# ------------------------------------------------------------
admission_type_mapping = extract_id_mapping(
    "admission_type_id"
)

discharge_disposition_mapping = extract_id_mapping(
    "discharge_disposition_id"
)

admission_source_mapping = extract_id_mapping(
    "admission_source_id"
)


# ============================================================
# Validate Parsed Administrative Mappings
# ============================================================

administrative_mappings = {
    "Admission Type": admission_type_mapping,
    "Discharge Disposition": discharge_disposition_mapping,
    "Admission Source": admission_source_mapping
}

for name, mapping in administrative_mappings.items():

    if not mapping:
        raise ValueError(
            f"{name} mapping could not be parsed."
        )

    print(
        f"{name} mappings parsed: {len(mapping)}"
    )

print(
    "\nUCI administrative ID mappings parsed successfully."
)

**Create Readable Administrative Category Labels**

In [ ]:
df["admission_type_label"] = (
    df["admission_type_id"]
    .map(admission_type_mapping)
    .fillna("Unknown / Unmapped")
)

df["discharge_disposition_label"] = (
    df["discharge_disposition_id"]
    .map(discharge_disposition_mapping)
    .fillna("Unknown / Unmapped")
)

df["admission_source_label"] = (
    df["admission_source_id"]
    .map(admission_source_mapping)
    .fillna("Unknown / Unmapped")
)

print("Readable administrative labels created successfully.")

**Create Target Variable**


In [ ]:
# Create binary target:
# 1 = Readmitted within 30 days
# 0 = Not readmitted within 30 days

df["readmitted_30"] = (
    df["readmitted"] == "<30"
).astype(int)

target_summary = pd.DataFrame({
    "Count": df["readmitted_30"].value_counts().sort_index(),
    "Percent": (
        df["readmitted_30"]
        .value_counts(normalize=True)
        .sort_index() * 100
    ).round(2)
})

target_summary.index = [
    "No 30-Day Readmission",
    "30-Day Readmission"
]

display(target_summary)

**Feature Labels Mapping**

In [ ]:
feature_labels = {
    "age": "Age Group",
    "time_in_hospital": "Time in Hospital",
    "num_lab_procedures": "Number of Lab Procedures",
    "num_procedures": "Number of Procedures",
    "num_medications": "Number of Medications",
    "number_outpatient": "Prior Outpatient Visits",
    "number_emergency": "Prior Emergency Visits",
    "number_inpatient": "Prior Inpatient Visits",
    "number_diagnoses": "Number of Diagnoses",
    "max_glu_serum": "Maximum Glucose Serum Result",
    "A1Cresult": "A1C Test Result",
    "change": "Medication Change",
    "diabetesMed": "Diabetes Medication Prescribed",
    "readmitted_30": "30-Day Readmission"
}

print("Readable feature labels created successfully.")

### **Exploratory Data Analysis**
This section explores patterns associated with 30-day hospital readmission before model development. The analysis examines the target distribution, numerical feature associations, demographic patterns, prior healthcare utilization, clinical-care characteristics, discharge disposition, and correlations among numerical variables.

Total encounters are also compared with unique patients to identify repeat encounters. These exploratory findings describe associations within the dataset and should not be interpreted as evidence of causation or as model-derived feature importance.

**Target Distribution and Numerical Feature Relevance**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ------------------------------------------------------------
# 1. Overall 30-Day Readmission Distribution
# ------------------------------------------------------------

ax1 = sns.countplot(
    data=df,
    x="readmitted_30",
    ax=axes[0]
)

axes[0].set_title(
    "Distribution of 30-Day Readmission",
    pad=15
)

axes[0].set_xlabel("Readmission Status")
axes[0].set_ylabel("Number of Encounters")

axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(
    [
        "No 30-Day Readmission",
        "30-Day Readmission"
    ]
)

counts = (
    df["readmitted_30"]
    .value_counts()
    .sort_index()
)

percentages = (
    df["readmitted_30"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

for i, bar in enumerate(ax1.patches):
    count = counts.iloc[i]
    percent = percentages.iloc[i]

    ax1.annotate(
        f"{count:,}\n({percent:.2f}%)",
        (
            bar.get_x() + bar.get_width() / 2,
            bar.get_height()
        ),
        ha="center",
        va="bottom",
        xytext=(0, 5),
        textcoords="offset points"
    )


# ------------------------------------------------------------
# 2. Numerical Feature Relevance
# ------------------------------------------------------------

numeric_cols = (
    df.select_dtypes(include=np.number)
    .columns
    .difference([
        "encounter_id",
        "patient_nbr",
        "readmitted_30",
        "admission_type_id",
        "discharge_disposition_id",
        "admission_source_id"
    ])
)

feature_relevance = (
    df[list(numeric_cols) + ["readmitted_30"]]
    .corr()["readmitted_30"]
    .drop("readmitted_30")
    .abs()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

feature_relevance.columns = [
    "Feature",
    "Absolute Correlation"
]

feature_relevance["Feature Label"] = (
    feature_relevance["Feature"]
    .map(feature_labels)
    .fillna(feature_relevance["Feature"])
)

ax2 = sns.barplot(
    data=feature_relevance,
    x="Absolute Correlation",
    y="Feature Label",
    ax=axes[1]
)

axes[1].set_title(
    "Top Numerical Features Associated with 30-Day Readmission",
    pad=15
)

axes[1].set_xlabel(
    "Absolute Correlation with Readmission"
)
axes[1].set_ylabel("Feature")

for container in ax2.containers:
    ax2.bar_label(
        container,
        fmt="%.3f",
        padding=3
    )

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Race-Level Readmission Summary
# ------------------------------------------------------------

race_readmission = (
    df.dropna(subset=["race"])
    .groupby("race", observed=False)["readmitted_30"]
    .agg(["count", "mean"])
    .reset_index()
)

# Convert readmission rate to percentage
race_readmission["Readmission Rate (%)"] = (
    race_readmission["mean"] * 100
)

# Create readable race summary table
race_summary = (
    race_readmission[
        ["race", "count", "Readmission Rate (%)"]
    ]
    .rename(
        columns={
            "race": "Race",
            "count": "Number of Encounters"
        }
    )
    .sort_values(
        "Readmission Rate (%)",
        ascending=False
    )
    .reset_index(drop=True)
)

print("30-Day Readmission Summary by Race")

display(
    race_summary.round({
        "Readmission Rate (%)": 2
    })
)

**Demographic Patterns in 30-Day Readmission**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ------------------------------------------------------------
# Age Group
# ------------------------------------------------------------

age_readmission = (
    df.groupby("age", observed=False)["readmitted_30"]
    .mean()
    .mul(100)
    .reset_index(name="Readmission Rate (%)")
)

sns.barplot(
    data=age_readmission,
    x="age",
    y="Readmission Rate (%)",
    ax=axes[0]
)

axes[0].set_title(
    "Readmission Rate by Age Group",
    pad=15
)
axes[0].set_xlabel("Age Group")
axes[0].set_ylabel("Readmission Rate (%)")
axes[0].tick_params(axis="x", rotation=45)


# ------------------------------------------------------------
# Race
# ------------------------------------------------------------

race_readmission = (
    df.dropna(subset=["race"])
    .groupby("race", observed=False)["readmitted_30"]
    .mean()
    .mul(100)
    .reset_index(name="Readmission Rate (%)")
)

sns.barplot(
    data=race_readmission,
    x="race",
    y="Readmission Rate (%)",
    ax=axes[1]
)

axes[1].set_title(
    "Readmission Rate by Race",
    pad=15
)
axes[1].set_xlabel("Race")
axes[1].set_ylabel("Readmission Rate (%)")
axes[1].tick_params(axis="x", rotation=35)


# ------------------------------------------------------------
# Gender
# ------------------------------------------------------------

gender_readmission = (
    df.groupby("gender", observed=False)["readmitted_30"]
    .mean()
    .mul(100)
    .reset_index(name="Readmission Rate (%)")
)

sns.barplot(
    data=gender_readmission,
    x="gender",
    y="Readmission Rate (%)",
    ax=axes[2]
)

axes[2].set_title(
    "Readmission Rate by Gender",
    pad=15
)
axes[2].set_xlabel("Gender")
axes[2].set_ylabel("Readmission Rate (%)")

plt.tight_layout()
plt.show()

**Prior Healthcare Utilization and Readmission**

In [ ]:
# Features representing healthcare utilization in the prior year
utilization_features = [
    "number_inpatient",
    "number_emergency",
    "number_outpatient"
]

# Calculate average prior visits for each readmission group
utilization_summary = []

for feature in utilization_features:

    no_readmission_mean = df.loc[
        df["readmitted_30"] == 0,
        feature
    ].mean()

    readmission_mean = df.loc[
        df["readmitted_30"] == 1,
        feature
    ].mean()

    utilization_summary.append({
        "Feature": feature_labels.get(feature, feature),
        "No 30-Day Readmission": no_readmission_mean,
        "30-Day Readmission": readmission_mean
    })

# Convert results to DataFrame
utilization_df = pd.DataFrame(utilization_summary)

# Reshape for grouped bar chart
utilization_long = utilization_df.melt(
    id_vars="Feature",
    var_name="Readmission Status",
    value_name="Average Prior Visits"
)

# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------

plt.figure(figsize=(11, 6))

ax = sns.barplot(
    data=utilization_long,
    x="Feature",
    y="Average Prior Visits",
    hue="Readmission Status"
)

plt.title(
    "Prior Healthcare Utilization by 30-Day Readmission Status",
    pad=15
)

plt.xlabel("Prior Healthcare Utilization")
plt.ylabel("Average Number of Visits")

# Add values above each bar
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f",
        padding=3
    )

# Place legend below the plot
plt.legend(
    title="Readmission Status",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    ncol=2,
    frameon=False
)

plt.tight_layout()
plt.show()

**Discharge Disposition and 30-Day Readmission**

In [ ]:
discharge_summary = (
    df.groupby(
        "discharge_disposition_label",
        observed=False
    )["readmitted_30"]
    .agg(["count", "mean"])
    .reset_index()
)

discharge_summary[
    "Readmission Rate (%)"
] = (
    discharge_summary["mean"]
    * 100
)

discharge_summary = (
    discharge_summary
    .rename(
        columns={
            "discharge_disposition_label":
                "Discharge Disposition",

            "count":
                "Number of Encounters"
        }
    )
)

# Select the 10 most common discharge destinations
top_dispositions = (
    discharge_summary
    .nlargest(
        10,
        "Number of Encounters"
    )
    .sort_values(
        "Readmission Rate (%)",
        ascending=False
    )
)

plt.figure(
    figsize=(11, 6)
)

ax = sns.barplot(
    data=top_dispositions,
    x="Readmission Rate (%)",
    y="Discharge Disposition"
)

plt.title(
    "30-Day Readmission Rate by Discharge Disposition",
    pad=15
)

plt.xlabel(
    "30-Day Readmission Rate (%)"
)

plt.ylabel(
    "Discharge Disposition"
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=3
    )

plt.tight_layout()
plt.show()

**Clinical Care and Testing Patterns**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ------------------------------------------------------------
# A1C Test Result
# ------------------------------------------------------------

a1c_readmission = (
    df.assign(
        A1C_Display=df["A1Cresult"].fillna("Not Tested")
    )
    .groupby("A1C_Display", observed=False)["readmitted_30"]
    .mean()
    .mul(100)
    .reset_index(name="Readmission Rate (%)")
)

sns.barplot(
    data=a1c_readmission,
    x="A1C_Display",
    y="Readmission Rate (%)",
    ax=axes[0]
)

axes[0].set_title(
    "30-Day Readmission Rate by A1C Test Result",
    pad=15
)
axes[0].set_xlabel("A1C Test Result")
axes[0].set_ylabel("Readmission Rate (%)")


# ------------------------------------------------------------
# Medication Change
# ------------------------------------------------------------

change_readmission = (
    df.groupby("change", observed=False)["readmitted_30"]
    .mean()
    .mul(100)
    .reset_index(name="Readmission Rate (%)")
)

sns.barplot(
    data=change_readmission,
    x="change",
    y="Readmission Rate (%)",
    ax=axes[1]
)

axes[1].set_title(
    "30-Day Readmission Rate by Medication Change",
    pad=15
)
axes[1].set_xlabel("Medication Change")
axes[1].set_ylabel("Readmission Rate (%)")

plt.tight_layout()
plt.show()

**Selected Numerical Feature Correlations**

In [ ]:
numeric_eda_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "readmitted_30"
]

correlation_matrix = (
    df[numeric_eda_features]
    .corr()
)

readable_corr_labels = [
    feature_labels.get(feature, feature)
    for feature in correlation_matrix.columns
]

plt.figure(figsize=(11, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    xticklabels=readable_corr_labels,
    yticklabels=readable_corr_labels
)

plt.title(
    "Correlation Matrix of Selected Numerical Features",
    pad=15
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

**Patient Encounter Assessment**



In [ ]:
total_encounters = len(df)
unique_patients = df["patient_nbr"].nunique()
repeat_encounters = total_encounters - unique_patients

print(f"Total Encounters:  {total_encounters:,}")
print(f"Unique Patients:   {unique_patients:,}")
print(f"Repeat Encounters: {repeat_encounters:,}")

**Patient Overlap Assessment**

In [ ]:
# ============================================================
# Patient Overlap Assessment
# ============================================================

repeat_patient_percent = (
    repeat_encounters
    / total_encounters
) * 100

print(
    f"Repeat Encounter Percentage: "
    f"{repeat_patient_percent:.2f}%"
)

### **Feature Preparation**
This section prepares the dataset for machine learning by separating the predictors from the 30-day readmission target. Encounter ID and patient ID are excluded because they are identifiers, while the original readmission variable is removed to prevent target leakage. Administrative labels created for EDA are also excluded to avoid duplicate feature representations.

The data is divided into stratified training and testing sets to preserve the imbalanced target distribution. Numerical and categorical features are then identified, with administrative ID variables treated as categorical features. Missing numerical values are imputed using the median, missing categorical values using the most frequent category, and categorical predictors are transformed using one-hot encoding.

**Define Features and Target**

In [ ]:
# ============================================================
# Define Features and Target
# ============================================================

# Administrative label columns are used only for readable EDA
# and are excluded from modeling to avoid duplicate representations.

X = df.drop(
    columns=[
        "encounter_id",
        "patient_nbr",
        "readmitted",
        "readmitted_30",
        "admission_type_label",
        "discharge_disposition_label",
        "admission_source_label"
    ]
)

y = df["readmitted_30"]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

**Train-Test Split**

In [ ]:
# ============================================================
# Stratified Train-Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Set:", X_train.shape)
print("Testing Set:", X_test.shape)

print("\nTraining Target Distribution:")
print(
    y_train
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

print("\nTesting Target Distribution:")
print(
    y_test
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

**Identify Numerical and Categorical Features**

In [ ]:
# These variables are stored numerically but represent categories
categorical_id_features = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

# True numerical features
numeric_features = (
    X.select_dtypes(include=np.number)
    .columns
    .difference(categorical_id_features)
    .tolist()
)

# Object/string categorical features
categorical_features = (
    X.select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

# Add categorical ID variables
categorical_features += categorical_id_features

print("Total Modeling Predictors:", X.shape[1])
print("Numerical Features:", len(numeric_features))
print("Categorical Features:", len(categorical_features))
print(
    "Total Identified:",
    len(numeric_features) + len(categorical_features)
)

**Create Preprocessing Pipeline**

In [ ]:
# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")

### **Initial Model Development**
This section develops an initial Random Forest classifier to predict 30-day hospital readmission. Random Forest can capture complex, nonlinear relationships among numerical and categorical predictors while supporting feature-importance analysis.

The model uses 100 decision trees and a fixed random state for reproducibility. Preprocessing and classification are combined within a single pipeline to ensure consistent transformation of the training and testing data. The resulting performance provides the initial benchmark for evaluating the effect of model optimization.

**Build and Train Initial Random Forest Model**

In [ ]:
random_forest_model = Pipeline(steps=[
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    )
])

random_forest_model.fit(
    X_train,
    y_train
)

print("Initial Random Forest model trained successfully.")

### **Model Evaluation**
This section evaluates the initial Random Forest using the training and held-out testing data. Training and testing accuracy are compared to assess generalization and potential overfitting.

Because the 30-day readmission target is highly imbalanced, performance is evaluated using accuracy, precision, recall, F1 score, and ROC-AUC. The classification report, confusion matrix, and ROC curve provide additional assessment of minority-class detection and overall discrimination. These results establish the initial performance benchmark for model optimization.

**Compare Training and Testing Accuracy**

In [ ]:
train_accuracy = random_forest_model.score(
    X_train,
    y_train
)

test_accuracy = random_forest_model.score(
    X_test,
    y_test
)

accuracy_gap = (
    train_accuracy - test_accuracy
)

print("Training vs. Testing Accuracy")
print("-" * 35)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy:  {test_accuracy:.4f}")
print(f"Accuracy Gap:      {accuracy_gap:.4f}")

**Evaluate Initial Model Performance**

In [ ]:
# Generate predictions
y_pred = random_forest_model.predict(
    X_test
)

y_prob = random_forest_model.predict_proba(
    X_test
)[:, 1]

# Calculate evaluation metrics
accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

# Performance table
model_performance = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]
})

display(
    model_performance.round(4)
)

**Initial Classification Report**

In [ ]:
print("Classification Report")
print("-" * 55)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "No 30-Day Readmission",
            "30-Day Readmission"
        ],
        zero_division=0
    )
)

**Initial Confusion Matrix and ROC Curve**

In [ ]:
# Create confusion matrix

cm = confusion_matrix(
    y_test,
    y_pred
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5)
)

# ------------------------------------------------------------
# Confusion Matrix
# ------------------------------------------------------------

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=True,
    ax=axes[0]
)

axes[0].set_title(
    "Random Forest Confusion Matrix",
    pad=15
)

axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

axes[0].set_xticklabels(
    ["No Readmission", "Readmission"]
)

axes[0].set_yticklabels(
    ["No Readmission", "Readmission"],
    rotation=0
)

# ------------------------------------------------------------
# ROC Curve
# ------------------------------------------------------------

RocCurveDisplay.from_predictions(
    y_test,
    y_prob,
    name="Random Forest",
    ax=axes[1]
)

axes[1].plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray"
)

axes[1].set_title(
    "Random Forest ROC Curve",
    pad=15
)

axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")

axes[1].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    frameon=False
)

fig.suptitle(
    "Initial Random Forest Model Evaluation",
    fontsize=14,
    y=1.03
)

plt.tight_layout()
plt.show()

### **Initial Feature Importance and Model Summary**
This section examines the predictors contributing most strongly to the fitted Random Forest model and summarizes its initial predictive performance. Feature importance reflects each predictor's contribution to the model's predictions and should not be interpreted as evidence of a causal relationship.

**Initial Feature Importance**

In [ ]:
# Extract transformed feature names
feature_names = (
    random_forest_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# Extract fitted classifier
rf_classifier = (
    random_forest_model
    .named_steps["classifier"]
)

# Create feature-importance DataFrame
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_classifier.feature_importances_
})

# ------------------------------------------------------------
# Convert model feature names to readable labels
# ------------------------------------------------------------

def make_readable_feature_name(feature_name):

    feature_name = (
        feature_name
        .replace("numeric__", "")
        .replace("categorical__", "")
    )

    # Direct feature match
    if feature_name in feature_labels:
        return feature_labels[feature_name]

    # One-hot encoded categorical feature
    for original_name, readable_name in feature_labels.items():

        prefix = original_name + "_"

        if feature_name.startswith(prefix):

            category = feature_name[len(prefix):]

            return f"{readable_name}: {category}"

    # Fallback
    return (
        feature_name
        .replace("_", " ")
        .title()
    )


feature_importance["Feature Label"] = (
    feature_importance["Feature"]
    .apply(make_readable_feature_name)
)

# Select Top 15
top_features = (
    feature_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .head(15)
)

# Plot
plt.figure(figsize=(10, 7))

ax = sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature Label"
)

plt.title(
    "Top 15 Random Forest Feature Importances",
    pad=15
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.3f",
        padding=3
    )

plt.tight_layout()
plt.show()

**Initial Model Summary**

In [ ]:
initial_model_summary = pd.DataFrame({
    "Model": [
        "Random Forest"
    ],
    "Training Accuracy": [
        train_accuracy
    ],
    "Testing Accuracy": [
        test_accuracy
    ],
    "Accuracy Gap": [
        accuracy_gap
    ],
    "Precision": [
        precision
    ],
    "Recall": [
        recall
    ],
    "F1 Score": [
        f1
    ],
    "ROC-AUC": [
        roc_auc
    ]
})

print(
    "Initial Random Forest Performance Summary"
)

display(
    initial_model_summary.round(4)
)

### **Model Optimization and Evaluation**

The initial Random Forest established a baseline for predicting 30-day readmission. Although the model achieved high overall accuracy, evaluation revealed substantial overfitting and poor identification of the minority readmission class.

This section refines the Random Forest through hyperparameter tuning, evaluates the resulting improvements, and investigates whether classification-threshold adjustment can further improve performance in a highly imbalanced healthcare prediction problem.

**Hyperparameter Optimization**

In [ ]:
# ============================================================
# Random Forest Hyperparameter Optimization
# ============================================================

from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV
)

# 5-fold stratified cross-validation
stratified_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Random Forest tuning pipeline
rf_tuning_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                random_state=42,
                n_jobs=1
            )
        )
    ]
)

# Hyperparameter search space
param_distributions = {
    "classifier__n_estimators": [
        100,
        150
    ],
    "classifier__max_depth": [
        8,
        12,
        16
    ],
    "classifier__min_samples_leaf": [
        2,
        5,
        10
    ],
    "classifier__class_weight": [
        "balanced",
        "balanced_subsample"
    ]
}

rf_search = RandomizedSearchCV(
    estimator=rf_tuning_model,
    param_distributions=param_distributions,
    n_iter=6,
    scoring="f1",
    cv=stratified_cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(
    X_train,
    y_train
)

print("\nBest Parameters:")
print(rf_search.best_params_)

print(
    f"\nBest Cross-Validated F1 Score: "
    f"{rf_search.best_score_:.4f}"
)

tuned_rf_model = rf_search.best_estimator_

**Evaluate Tuned Random Forest**

In [ ]:
# ============================================================
# Evaluate Tuned Random Forest
# ============================================================

tuned_pred = tuned_rf_model.predict(
    X_test
)

tuned_prob = (
    tuned_rf_model.predict_proba(
        X_test
    )[:, 1]
)

tuned_accuracy = accuracy_score(
    y_test,
    tuned_pred
)

tuned_precision = precision_score(
    y_test,
    tuned_pred,
    zero_division=0
)

tuned_recall = recall_score(
    y_test,
    tuned_pred,
    zero_division=0
)

tuned_f1 = f1_score(
    y_test,
    tuned_pred,
    zero_division=0
)

tuned_roc_auc = roc_auc_score(
    y_test,
    tuned_prob
)

tuned_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    "Score": [
        tuned_accuracy,
        tuned_precision,
        tuned_recall,
        tuned_f1,
        tuned_roc_auc
    ]
})

display(
    tuned_results.round(4)
)

**Initial vs Tuned Comparison**

In [ ]:
# ====================================================
# Initial vs Tuned Comparison
# ====================================================

# Calculate missing metrics for the tuned model
tuned_train_accuracy = tuned_rf_model.score(X_train, y_train)
tuned_test_accuracy = tuned_rf_model.score(X_test, y_test)
tuned_accuracy_gap = tuned_train_accuracy - tuned_test_accuracy

comparison_df = pd.DataFrame({
    "Metric": [
        "Training Accuracy",
        "Testing Accuracy",
        "Accuracy Gap",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    "Initial Model": [
        train_accuracy,
        test_accuracy,
        accuracy_gap,
        precision,
        recall,
        f1,
        roc_auc
    ],
    "Tuned Model": [
        tuned_train_accuracy,
        tuned_test_accuracy,
        tuned_accuracy_gap,
        tuned_precision,
        tuned_recall,
        tuned_f1,
        tuned_roc_auc
    ]
})

comparison_df["Improvement"] = (
    comparison_df["Tuned Model"]
    - comparison_df["Initial Model"]
)

display(
    comparison_df.round(4)
)

### **Precision-Recall Analysis**

Although hyperparameter tuning improved recall and F1 score, the default classification threshold of 0.50 may not be optimal for an imbalanced readmission prediction problem.

This section evaluates the precision-recall tradeoff across probability thresholds and identifies the threshold that maximizes F1 score.

**Precision-Recall Curve**

In [ ]:
# ============================================================
# Precision-Recall Curve
# ============================================================

from sklearn.metrics import (
    PrecisionRecallDisplay
)

plt.figure(figsize=(7, 5))

PrecisionRecallDisplay.from_predictions(
    y_test,
    tuned_prob
)

plt.title(
    "Precision-Recall Curve",
    pad=15
)

plt.tight_layout()
plt.show()

**Threshold Optimization**

In [ ]:
# ============================================================
# Identify Optimal Threshold
# ============================================================

from sklearn.metrics import (
    precision_recall_curve
)

precision_values, recall_values, thresholds = (
    precision_recall_curve(
        y_test,
        tuned_prob
    )
)

f1_scores = (
    2
    * precision_values[:-1]
    * recall_values[:-1]
) / (
    precision_values[:-1]
    + recall_values[:-1]
)

best_idx = np.argmax(
    f1_scores
)

best_threshold = thresholds[
    best_idx
]

print(
    f"Best Threshold: "
    f"{best_threshold:.4f}"
)

print(
    f"Best F1 Score: "
    f"{f1_scores[best_idx]:.4f}"
)

**Evaluate Optimized Threshold**

In [ ]:
# ============================================================
# Evaluate Optimized Threshold
# ============================================================

threshold_pred = (
    tuned_prob >= best_threshold
).astype(int)

threshold_precision = precision_score(
    y_test,
    threshold_pred,
    zero_division=0
)

threshold_recall = recall_score(
    y_test,
    threshold_pred,
    zero_division=0
)

threshold_f1 = f1_score(
    y_test,
    threshold_pred,
    zero_division=0
)

threshold_results = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Score": [
        threshold_precision,
        threshold_recall,
        threshold_f1
    ]
})

display(
    threshold_results.round(4)
)

**Default vs Optimized Threshold**

In [ ]:
# ============================================================
# Threshold Comparison
# ============================================================

threshold_comparison = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Default Threshold": [
        tuned_precision,
        tuned_recall,
        tuned_f1
    ],
    "Optimized Threshold": [
        threshold_precision,
        threshold_recall,
        threshold_f1
    ]
})

threshold_comparison["Change"] = (
    threshold_comparison["Optimized Threshold"]
    - threshold_comparison["Default Threshold"]
)

display(
    threshold_comparison.round(4)
)

### **Final Optimization Summary**

In [ ]:
# ============================================================
# Final Optimization Summary
# ============================================================

summary_df = pd.DataFrame({
    "Category": [
        "Initial Recall",
        "Tuned Recall",
        "Threshold Recall",
        "Initial F1",
        "Tuned F1",
        "Threshold F1",
        "Initial ROC-AUC",
        "Tuned ROC-AUC"
    ],
    "Value": [
        round(recall, 4),
        round(tuned_recall, 4),
        round(threshold_recall, 4),
        round(f1, 4),
        round(tuned_f1, 4),
        round(threshold_f1, 4),
        round(roc_auc, 4),
        round(tuned_roc_auc, 4)
    ]
})

display(summary_df)

In [ ]:
# CELL 7: Threshold Tuning and Model Evaluation
# Predict probabilities on the test set
y_probs = best_model.predict_proba(X_test)[:, 1]

# Generate Precision-Recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_probs)

# Find the threshold that guarantees a minimum recall (e.g., 0.80) to catch high-risk patients
target_recall = 0.80
optimal_idx = np.where(recall >= target_recall)[0][-1]
optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else thresholds[-1]

print(f"Optimal Threshold for {target_recall*100}% Recall: {optimal_threshold:.4f}")

# Apply custom threshold
y_pred_custom = (y_probs >= optimal_threshold).astype(int)

print("\n--- Classification Report (Custom Threshold) ---")
print(classification_report(y_test, y_pred_custom))

# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred_custom)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix (Optimized Threshold)')
plt.xlabel('Predicted (0 = No, 1 = Readmitted)')
plt.ylabel('Actual (0 = No, 1 = Readmitted)')
plt.show()

In [ ]:
# CELL 8: Fairness and Bias Evaluation (Demographic Stratification)
# Evaluate recall and false positive rate across different racial demographics
if 'race' in X_test.columns:
    results_df = X_test.copy()
    results_df['Actual'] = y_test
    results_df['Predicted'] = y_pred_custom

    print("\n--- Model Performance by Race ---")
    for race_category in results_df['race'].dropna().unique():
        subset = results_df[results_df['race'] == race_category]

        if len(subset) > 0:
            actual = subset['Actual']
            pred = subset['Predicted']

            # Calculate True Positive Rate (Recall) and False Positive Rate
            tn, fp, fn, tp = confusion_matrix(actual, pred, labels=[0, 1]).ravel()
            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

            print(f"Race: {race_category: <15} | Recall (TPR): {tpr:.2f} | FPR: {fpr:.2f} | Samples: {len(subset)}")
else:
    print("Race column not found in test set for bias evaluation.")

### **Key Findings**

Hyperparameter optimization substantially improved minority-class detection and reduced overfitting relative to the initial Random Forest model. Recall increased from 0.0048 to 0.5764, while F1 score improved from 0.0096 to 0.2810 and ROC-AUC increased from 0.6574 to 0.6810.

Precision-recall analysis and threshold optimization demonstrated that additional improvements in F1 score are possible by adjusting the classification threshold, although such adjustments introduce trade-offs between precision and recall.

Further discussion of operational, ethical, and deployment considerations is provided in the accompanying written report.